Offset parameters phase predictor
--

Predict phases with offset parameters (angle, etc.).

Two options:
1. Compare the generation with the predicted phases and the ORIGINAL parameters to the original image.
   Take the original params when generating the image with the predicted phase so that the error of that image is only being caused by the predicted phase and not by any other parameters.
2. Also save image with offset parameters to see whether model is actually closer than with original params.


# Import modules


In [32]:
import numpy as np
import torch
from IPython.display import display

from src.dataset.generate_wave import SuperWave, WaveParams, wave_series_to_imgs
from src.phase_predictor.phase_predictor import PhasePredictor


# Create sample data


In [33]:
def mse(pred, true):
    '''
    ((pred - true) ** 2).mean()
    '''
    return ((pred - true) ** 2).mean()

wave_params = (
    WaveParams(
        resolution_px_per_km=2,
        resolution_seconds_per_time_step=30,
        angle=150,
        wavelength=80.0,
        amplitude=3,
        phase=np.pi,
        period=10,
        img_shape=(256, 256),
        timestep=0,
        alpha=0.8
    ),
    WaveParams(
        resolution_px_per_km=2,
        resolution_seconds_per_time_step=30,
        angle=80,
        wavelength=20.0,
        amplitude=7,
        phase=0.5 * np.pi,
        period=4,
        img_shape=(256, 256),
        timestep=0,
        alpha=0.5
    ),
    WaveParams(
        resolution_px_per_km=2,
        resolution_seconds_per_time_step=30,
        angle=20,
        wavelength=60.0,
        amplitude=4,
        phase=5,
        period=8,
        img_shape=(256, 256),
        timestep=0,
        alpha=0.3
    )
)

original_wave = SuperWave(
    wave_params=wave_params,
    time_steps=7,
    image_shape=(256, 256),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    to_cpu=False,
    to_numpy=False
)

# modify wave_params to create approximated wave
wave_params_approx = (
    WaveParams(
        resolution_px_per_km=2,
        resolution_seconds_per_time_step=30,
        angle=160,
        wavelength=75.0,
        amplitude=2.8,
        phase=0,
        period=9,
        img_shape=(256, 256),
        timestep=0,
        alpha=0.8
    ),
    WaveParams(
        resolution_px_per_km=2,
        resolution_seconds_per_time_step=30,
        angle=80,
        wavelength=20.0,
        amplitude=7,
        phase=0,
        period=4,
        img_shape=(256, 256),
        timestep=0,
        alpha=0.5
    ),
    WaveParams(
        resolution_px_per_km=2,
        resolution_seconds_per_time_step=30,
        angle=20,
        wavelength=60.0,
        amplitude=4,
        phase=0,
        period=8,
        img_shape=(256, 256),
        timestep=0,
        alpha=0.3
    )
)

approx_wave = SuperWave(
    wave_params=wave_params_approx,
    time_steps=7,
    image_shape=(256, 256),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    to_cpu=False,
    to_numpy=False
)

# %% Initialize PhasePredictor
wave_predictor = PhasePredictor(
    original_wave=original_wave,
    approximated_wave=approx_wave,
    loss_func=mse,
    initial_phases=None,
    learning_rate=10e-3,
    early_stopping=None,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)


# Fit the model and compute difference


In [34]:
# predict phases
steps = 1000

result = wave_predictor.fit(steps=steps)

# compute difference of fitted wave to original wave
diff = wave_predictor.difference()

# scale image when displaying
max_val, min_val = float(original_wave.wave_img.max()), float(original_wave.wave_img.min())

# compute difference between approximated wave and original wave
unfitted_diff = original_wave.wave_img - approx_wave.wave_img


Step 0     | Loss = 3.891218e+01 | Phases = [0.009999998845160007, 0.009999999776482582, -0.010000000707805157]


Step 10    | Loss = 3.614931e+01 | Phases = [0.11076678335666656, 0.10998190939426422, -0.10984399914741516]


Step 20    | Loss = 3.341600e+01 | Phases = [0.21458038687705994, 0.20977288484573364, -0.2088598757982254]


Step 30    | Loss = 3.074770e+01 | Phases = [0.3228680193424225, 0.30897170305252075, -0.3061355650424957]


Step 40    | Loss = 2.818158e+01 | Phases = [0.43605953454971313, 0.4070177376270294, -0.40068238973617554]


Step 50    | Loss = 2.575242e+01 | Phases = [0.5539162755012512, 0.5032733082771301, -0.4915241301059723]


... (90 outputs left out)

Step 960   | Loss = 9.513826e+00 | Phases = [3.12174391746521, 1.5692378282546997, -1.171604871749878]


Step 970   | Loss = 9.513826e+00 | Phases = [3.12174391746521, 1.5692378282546997, -1.171604871749878]


Step 980   | Loss = 9.513826e+00 | Phases = [3.12174391746521, 1.5692378282546997, -1.171604871749878]


Step 990   | Loss = 9.513826e+00 | Phases = [3.12174391746521, 1.5692378282546997, -1.171604871749878]


# Convert wave series to images


In [36]:
fitted_imgs = wave_series_to_imgs(
    wave_series=diff,
    scale='cut',
    match_range=(min_val, max_val),
    coloured=True
)

unfitted_imgs = wave_series_to_imgs(
    wave_series=unfitted_diff,
    scale='cut',
    match_range=(min_val, max_val),
    coloured=True
)


# Display images


In [37]:
print("Fitted wave difference to original wave:")
for i in fitted_imgs:
    display(i)
print("Unfitted wave difference to original wave:")
for i in unfitted_imgs:
    display(i)


Fitted wave difference to original wave:


Unfitted wave difference to original wave:
